In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.colors as mcolors
import networkx as nx
import copy

In [ ]:
# ==============================
# PARÂMETROS GERAIS
# ==============================

L = 100
n_lagartos = L**2
estrategias = ['O', 'Y', 'B']

b = 2
c = 0.5

matriz_payoff = np.array([[1, c, b],
                          [b, 1, c],
                          [c, b, 1]])

index_map = {'O': 0, 'Y': 1, 'B': 2}

n_geracoes = 500
n_pop = 1
LN = int(input("Valor de LN: "))
IN = int(input("Valor de IN: "))
z_B = 8
sobreposicao = "n"
fermi = input("Fermi? (s/n): ")
if fermi.lower() == 's':
    K = 0.001
else:
    K = None


output_dir = f"C:/Unicamp/mestrado/simulacoes/main/RPS-neighborhood/outputs/network/sensibility/"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# ==============================
# CLASSE LAGARTO
# ==============================

class Lagarto:
    def __init__(self, i, j, estrategia, fitness,
                 coord_vizinhos, n_vizinhos_LN, n_vizinhos_IN):
        self.i = i
        self.j = j
        self.estrategia = estrategia
        self.fitness = 0.0
        self.coord_vizinhos_LN = []
        self.coord_vizinhos_IN = []
        self.n_vizinhos_LN = n_vizinhos_LN
        self.n_vizinhos_IN = n_vizinhos_IN

    def calcular_coord_vizinhos(self, L, tipo):
        if tipo == "LN":
            n = self.n_vizinhos_LN
        elif tipo == "IN":
            n = self.n_vizinhos_IN

        if n <= 0:
            print("Erro: n_vizinhos <= 0")
            return

        i0, j0 = self.i, self.j

        def moore(r):
            return {
                ((i0 + dx) % L, (j0 + dy) % L)
                for dx in range(-r, r + 1)
                for dy in range(-r, r + 1)
                if not (dx == 0 and dy == 0)
            }

        def von_neumann(r):
            return {
                ((i0 + dx) % L, (j0 + dy) % L)
                for dx in range(-r, r + 1)
                for dy in range(-r, r + 1)
                if (dx != 0 or dy != 0) and (abs(dx) + abs(dy) <= r)
            }
        
        r = int(np.ceil((np.sqrt(1 + n) - 1) / 2))

        vn_r = von_neumann(r)
        mo_r = moore(r)

        if n <= 2 * r * (r + 1):    # faixa Moore(r-1) -> VN(r)
            base = moore(r - 1) if r > 1 else set()
            pool = list(vn_r - base)
        else:                       # faixa VN(r) -> Moore(r)
            base = vn_r
            pool = list(mo_r - base)

        faltam = n - len(base)
        if faltam <= 0:
            extras = []
        else:
            idx = np.random.choice(len(pool), size=faltam, replace=False)
            extras = [pool[k] for k in idx]

        if tipo == "LN":
            self.coord_vizinhos_LN = list(base) + extras
        elif tipo == "IN":
            self.coord_vizinhos_IN = list(base) + extras

    def mortalidade(self, A, w):
        return 1 / (1 + A * np.exp(w * self.fitness))

    def calcular_fitness_rede(self, G):
        fitness_total = 0.0
        for vizinho in vizinhos_unicos_rede(G, self):
            fitness_total += matriz_payoff[index_map[self.estrategia], index_map[vizinho.estrategia]]
        self.fitness = fitness_total

    def atualizar_links_lagarto(self, G, L, mapa, tipo):
        G.remove_edges_from(list(G.out_edges(self)))

        self.calcular_coord_vizinhos(L, tipo)

        for (ni, nj) in self.coord_vizinhos_LN if tipo == "LN" else self.coord_vizinhos_IN:
            vizinho = mapa[(ni, nj)]
            G.add_edge(self, vizinho)

    def fermi_update(self, G_IN, G_LN, K):
        lista_vizinhos = vizinhos_unicos_rede(G_LN, self)

        vizinho_escolhido = np.random.choice(lista_vizinhos)
        vizinho_escolhido.calcular_fitness_rede(G_IN)
        #print(f"Vizinho escolhido: {vizinho_escolhido.i}, {vizinho_escolhido.j}, {vizinho_escolhido.estrategia} com fitness {vizinho_escolhido.fitness}")
        
        prob_adotar = 1 / (1 + np.exp(- (vizinho_escolhido.fitness - self.fitness) / K))
        #print(f"Probabilidade de adotar: {prob_adotar:.4f}")

        if np.random.rand() < prob_adotar:
            #print(f"Adotou a estratégia do vizinho")
            return vizinho_escolhido.estrategia, vizinho_escolhido.n_vizinhos_LN, vizinho_escolhido.n_vizinhos_IN
        return self.estrategia, self.n_vizinhos_LN, self.n_vizinhos_IN
    
    def adaptative_update(self, G_IN, G_LN):
        lista_vizinhos = vizinhos_unicos_rede(G_LN, self)
        for vizinho in lista_vizinhos:
            vizinho.calcular_fitness_rede(G_IN)
        melhor_vizinho = max(lista_vizinhos, key=lambda v: v.fitness)
        if melhor_vizinho.fitness > self.fitness:
            return melhor_vizinho.estrategia, melhor_vizinho.n_vizinhos_LN, melhor_vizinho.n_vizinhos_IN
        return self.estrategia, self.n_vizinhos_LN, self.n_vizinhos_IN

# ==============================
# FUNÇÕES AUXILIARES
# ==============================

def vizinhos_unicos_rede(G, no):
    return list(set(G.predecessors(no)).union(set(G.successors(no))))

def criar_lagartos():
    lista = []
    for i in range(L):
        for j in range(L):
            estrategia = np.random.choice(estrategias)
            n_vizinhos_LN = LN
            n_vizinhos_IN = IN
            lista.append(Lagarto(i, j, estrategia, 0, [], n_vizinhos_LN, n_vizinhos_IN))
    return lista

def calcular_freq_rede(G, estrategias):
    n = G.number_of_nodes()
    if n == 0:
        return np.zeros(len(estrategias), dtype=float)

    contagem = {e: 0 for e in estrategias}
    for lagarto in G.nodes:
        contagem[lagarto.estrategia] += 1

    return np.array([contagem[e] / n for e in estrategias], dtype=float)

def grau_unico(G, l):
    return len(set(G.predecessors(l)).union(set(G.successors(l))))

def media_vizinhos_rede(G, lista_lagartos):
    graus = [grau_unico(G, l) for l in lista_lagartos]
    return np.mean(graus) if len(graus) > 0 else np.nan

def media_vizinhos_por_estrategia_rede(G, lista_lagartos):
    medias = []
    for e in estrategias:
        graus = [grau_unico(G, l) for l in lista_lagartos if l.estrategia == e]
        medias.append(np.mean(graus) if len(graus) > 0 else np.nan)
    return medias

In [ ]:
# ==============================
# SIMULAÇÃO
# ==============================

def simulacao(fermi, K, sobreposicao, seed=None):
    resultados = []

    for pop in range(n_pop):
        if seed is not None:
            np.random.seed(seed + pop)
            
        G_LN = nx.DiGraph()
        G_IN = nx.DiGraph()

        lista_lagartos = criar_lagartos()
        mapa = {(l.i, l.j): l for l in lista_lagartos}

        for lagarto in lista_lagartos:
            G_LN.add_node(lagarto)
            G_IN.add_node(lagarto)

        for lagarto in lista_lagartos:
            lagarto.atualizar_links_lagarto(G_LN, L, mapa, "LN")
            lagarto.atualizar_links_lagarto(G_IN, L, mapa, "IN")

        redes_geracoes = [copy.deepcopy(G_LN), copy.deepcopy(G_IN)]
        
        matriz_posicao = np.empty((L, L), dtype=object)
        for l in lista_lagartos:
            matriz_posicao[l.i, l.j] = l.estrategia

        freq = calcular_freq_rede(G_IN, estrategias)
        vizinhos_mean_LN = media_vizinhos_rede(G_LN, lista_lagartos)
        vizinhos_mean_IN = media_vizinhos_rede(G_IN, lista_lagartos)
        r_por_estrat = media_vizinhos_por_estrategia_rede(G_IN, lista_lagartos)

        resultados.append({
            "pop": pop,
            "t": -1,
            "freq_O": freq[0],
            "freq_Y": freq[1],
            "freq_B": freq[2],
            "vizinhos_mean_LN": vizinhos_mean_LN,
            "vizinhos_mean_IN": vizinhos_mean_IN,
            "sobreposicao": sobreposicao,
            "fermi": fermi,
            "K": K
        })

        for t in range(n_geracoes):
            print(f"População {pop+1} - Geração {t+1}/{n_geracoes}")

            mudancas = []
            if fermi.lower() == 's':
                for lagarto in lista_lagartos:
                    lagarto.calcular_fitness_rede(G_IN)
                    estrategia_escolhida, LN_escolhida, IN_escolhida = lagarto.fermi_update(G_IN, G_LN, K)
                    mudancas.append((lagarto, estrategia_escolhida, LN_escolhida, IN_escolhida))
            
            else:
                for lagarto in lista_lagartos:
                    lagarto.calcular_fitness_rede(G_IN)
                    estrategia_escolhida, LN_escolhida, IN_escolhida = lagarto.adaptative_update(G_IN, G_LN)
                    mudancas.append((lagarto, estrategia_escolhida, LN_escolhida, IN_escolhida))
            
            for lagarto, estrategia_escolhida, LN_escolhida, IN_escolhida in mudancas:
                    lagarto.estrategia = estrategia_escolhida
                    lagarto.n_vizinhos_LN = LN_escolhida
                    lagarto.n_vizinhos_IN = IN_escolhida
                    lagarto.atualizar_links_lagarto(G_LN, L, mapa, "LN")
                    lagarto.atualizar_links_lagarto(G_IN, L, mapa, "IN")
            
            redes_geracoes.append(copy.deepcopy(G_LN), copy.deepcopy(G_IN))

            for l in lista_lagartos:
                matriz_posicao[l.i, l.j] = l.estrategia

            freq = calcular_freq_rede(G_IN, estrategias)
            vizinhos_mean_LN = media_vizinhos_rede(G_LN, lista_lagartos)
            vizinhos_mean_IN = media_vizinhos_rede(G_IN, lista_lagartos)
            r_por_estrat = media_vizinhos_por_estrategia_rede(G_IN, lista_lagartos)

            resultados.append({
            "pop": pop,
            "t": t,
            "freq_O": freq[0],
            "freq_Y": freq[1],
            "freq_B": freq[2],
            "vizinhos_mean_LN": vizinhos_mean_LN,
            "vizinhos_mean_IN": vizinhos_mean_IN,
            "sobreposicao": sobreposicao,
            "fermi": fermi,
            "K": K
            })

    return pd.DataFrame(resultados)